# Unreleased Feats

## Note
Please use the colab notebook for this, that will make the installation of different branches of the repo more fast and easily accessible

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/phoeenniixx/pytorch-forecasting-v2-user-testing/blob/main/notebooks/02_unreleased.ipynb)

Lets try out some feats from the main which are yet to be released

## Installation
Lets first install the `pytorch-forecasting` present on the main

In [ ]:
!pip install git+https://github.com/sktime/pytorch-forecasting.git

Some unreleased feats:
- scalers and normalizer support in `EncoderDecoderTimeSeriesDataModule`
- Few more models (like `DecoderMLP`)

In [ ]:
from pytorch_forecasting.data.data_module import EncoderDecoderTimeSeriesDataModule
from pytorch_forecasting.data.timeseries import TimeSeries

In [3]:
# Copied here from utils to prevent any issues in colab!
import numpy as np
import pandas as pd
def load_toydata(num_series, seq_length):
    data_list = []
    for i in range(num_series):
        x = np.arange(seq_length)
        level = 10 ** (i % 3)
        y = level * np.sin(x / 5.0) + np.random.normal(scale=0.1, size=seq_length)
        category = i % 5
        static_value = np.random.rand()
        for t in range(seq_length - 1):
            data_list.append(
                {
                    "series_id": i,
                    "time_idx": t,
                    "x": y[t],
                    "y": y[t + 1],
                    "category": category,
                    "future_known_feature": np.cos(t / 10),
                    "static_feature": static_value,
                    "static_feature_cat": i % 3,
                }
            )
    data_df = pd.DataFrame(data_list)
    return data_df

In [ ]:
# Play around with the toy dataset
num_series = 100 
seq_length = 50 
df = load_toydata(num_series, seq_length)
df.head()

Create D1 layer

In [ ]:
dataset = TimeSeries(
    # TODO: specify the required arguments to create a TimeSeries dataset
)

## Scalers

Try `EncoderDecoderTimeSeriesDataModule` with different scalers options: `None` or the supported scalers

Supported scalers
 * **PyTorch Forecasting Normalizers**:

     * `pytorch_forecasting.data.encoders.TorchNormalizer`
     * `pytorch_forecasting.data.encoders.GroupNormalizer`
     * `pytorch_forecasting.data.encoders.EncoderNormalizer`

 * **Scikit-Learn Scalers**:

     * ``StandardScaler``
     * ``RobustScaler``
     * ``MinMaxScaler``
     * ``MaxAbsScaler``

In [5]:
from sklearn.preprocessing import StandardScaler, RobustScaler, MinMaxScaler, MaxAbsScaler
from pytorch_forecasting.data.encoders import GroupNormalizer, EncoderNormalizer, TorchNormalizer

In [ ]:
dm_1 = EncoderDecoderTimeSeriesDataModule(
    time_series_dataset=dataset,
    max_encoder_length=30,
    max_prediction_length=6,
    scalers={
        # TODO: specify the required scalers for the dataset per feature - 
        # see the raw dataset to determine which features you want to scale
    },
)

### Create multiple Datamodules with different scalers to compare the mean and std between them

In [ ]:
import torch
def first_batch(dm, seed=42):
    torch.manual_seed(seed)      # split uses randperm — seed for a like-for-like compare
    dm.setup(stage="fit")
    return next(iter(dm.train_dataloader()))

Code snippet to compare two dm's

In [ ]:
x_1, _ = first_batch(dm_1)
x_2, _ = first_batch(dm_2)
cont_names = [
    dm_2.time_series_metadata["cols"]["x"][i]
    for i in dm_2.continuous_indices
]

print(f"{'feature':<24}{'raw mean':>10}{'raw std':>10} | {'scaled mean':>12}{'scaled std':>11}")
for i, name in enumerate(cont_names):
    r = x_1["encoder_cont"][:, :, i]
    s = x_2["encoder_cont"][:, :, i]
    print(f"{name:<24}{r.mean():>10.3f}{r.std():>10.3f} | {s.mean():>12.3f}{s.std():>11.3f}")

To see the scalers that is being used

In [ ]:
adapter = dm_1._scalers["x"]
print(type(adapter._scaler).__name__)
print("mean:", adapter._scaler.mean_, "scale:", adapter._scaler.scale_)

## Target Normalizers

We can also normalize the targets - by passing `target_normalizer` to `EncoderDecoderTimeSeriesDataModule`:

- By default, no normalization
- Can pass `"auto"` to automatically choose a `target_normalizer` which best suites your data
- Or Pass the target_normalizer explicitly
  - Supported target_normalizers
      - `pytorch_forecasting.data.encoders.TorchNormalizer`,
      - `pytorch_forecasting.data.encoders.GroupNormalizer`,
      - `pytorch_forecasting.data.encoders.NaNLabelEncoder`,
      - `pytorch_forecasting.data.encoders.EncoderNormalizer`
- In case of multi-target, pass a list of normalizers for each target

In [6]:
# TODO: try different datasets to see and max_encoder_length and max_prediction_length to see how the target normalizers change for `auto` mode


### `target_normalizer="auto"` picks:

| Condition | Result |
|---|---|
| categorical target | `NaNLabelEncoder` |
| `max_encoder_length > 20` and `min_encoder_length > 1` | `EncoderNormalizer` |
| else + `group` set | `GroupNormalizer` |
| else | `TorchNormalizer` |
| positive, skew > 2.5 | `transformation="log"` |
| positive, skew ≤ 2.5 | `"relu"` |
| has negatives | no transformation |
| multiple targets | wrapped in `MultiNormalizer` |

In [ ]:
dm = EncoderDecoderTimeSeriesDataModule(
    time_series_dataset=dataset,
    max_encoder_length=30,
    max_prediction_length=12,
    target_normalizer="auto",
)

In [ ]:
dm.setup(stage="fit")
adapter = dm._target_normalizer
scaler = adapter._scaler
name = type(scaler).__name__ if scaler is not None else "None (no normalization)"
print(f"{name:<20} ")

## Task:
Try using `pkg` class with the `DecoderMLP` and datamodules with different Datamodule configs

As the model is compatible with `EncoderDecoderTimeSeriesDataModule`, you can apply preprocessing to the data before passing it to the models!

In [ ]:
from pytorch_forecasting.models.mlp import DecoderMLP_pkg_v2